# Question 3: CycleGAN — Domain Adaptation & Unpaired Image-to-Image Translation
## Sketch ↔ Photo Translation

This notebook implements **CycleGAN** for unpaired image-to-image translation between two domains:
- **Domain A** – Sketches (TU-Berlin / Sketchy / QuickDraw)
- **Domain B** – Photos

### Key Features
| Feature | Detail |
|---|---|
| Generator | ResNet-based (6 blocks) |
| Discriminator | PatchGAN |
| Image Size | 128 × 128 |
| Mixed Precision | ✓ `torch.amp` |
| Dual GPU | ✓ `DataParallel` |
| Losses | Adversarial + Cycle (λ=10) + Identity (λ=5) |
| Metrics | SSIM, PSNR |

---
## 1 · Environment & Imports

In [ ]:
# ── Install any missing packages (safe to re-run) ───────────────────────────
!pip install -q scikit-image tqdm matplotlib pillow

In [ ]:
import os
import glob
import random
import itertools
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

import warnings
warnings.filterwarnings('ignore')

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f'Device : {device}')
print(f'GPU(s) : {NUM_GPUS}')
if torch.cuda.is_available():
    for i in range(NUM_GPUS):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

---
## 2 · Reproducibility & Hyper-parameters

In [ ]:
# ── Seed ──────────────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Hyper-parameters ─────────────────────────────────────────────────────────
IMAGE_SIZE     = 128
BATCH_SIZE     = 4        # 4-8 for T4×2
NUM_WORKERS    = 2
NGF            = 64       # Generator filters
NDF            = 64       # Discriminator filters
RESNET_BLOCKS  = 6        # 6 (optimised for Kaggle)
NUM_EPOCHS     = 50
LR             = 0.0002
BETAS          = (0.5, 0.999)
LAMBDA_CYCLE   = 10.0
LAMBDA_IDENTITY = 5.0
DECAY_START    = 25       # Epoch to start linear LR decay
MAX_DATASET    = 5000     # Subset cap per domain
BUFFER_SIZE    = 50       # Replay buffer size
CKPT_INTERVAL  = 5        # Save checkpoint every N epochs

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT      = '/kaggle/input/'
OUTPUT_DIR     = '/kaggle/working/cyclegan_outputs/'
CHECKPOINT_DIR = '/kaggle/working/cyclegan_checkpoints/'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Image size : {IMAGE_SIZE}×{IMAGE_SIZE}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Epochs     : {NUM_EPOCHS}')
print(f'LR         : {LR}')
print(f'ResBlocks  : {RESNET_BLOCKS}')

---
## 3 · Data Preparation

In [ ]:
# ── 3.1  Discover datasets under /kaggle/input/ ─────────────────────────────
def discover_datasets(root):
    """Walk root dir and return {name: path} for each top-level folder."""
    found = {}
    if os.path.exists(root):
        for item in sorted(os.listdir(root)):
            p = os.path.join(root, item)
            if os.path.isdir(p):
                found[item] = p
    return found

datasets = discover_datasets(DATA_ROOT)
for name, path in datasets.items():
    print(f'  📁  {name:40s}  →  {path}')
print(f'\nTotal datasets: {len(datasets)}')

In [ ]:
# ── 3.2  Helper: collect image paths recursively ────────────────────────────
IMG_EXTS = {'*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp',
            '*.JPG', '*.JPEG', '*.PNG', '*.BMP', '*.WEBP'}

def collect_images(directory, max_images=None):
    """Return sorted list of image paths found recursively."""
    paths = []
    for ext in IMG_EXTS:
        paths.extend(glob.glob(os.path.join(directory, '**', ext), recursive=True))
    paths = sorted(set(paths))          # deterministic order
    if max_images and len(paths) > max_images:
        random.shuffle(paths)           # randomise before truncation
        paths = paths[:max_images]
    return paths

print('collect_images helper ready')

In [ ]:
# ── 3.3  Resolve Domain A (Sketches) and Domain B (Photos) ──────────────────
#
# Heuristic: match folder names against known keywords.
# ‣ For the **Sketchy dataset** the structure is:
#      sketchy-dataset/
#          sketch/  ← Domain A
#          photo/   ← Domain B
#
# ‣ For TU-Berlin the HuggingFace repo stores PNGs directly.
# ‣ For QuickDraw the images are stored as simplified drawings.
#
# Override the auto-detection if needed.

SKETCH_KEYWORDS = ['sketch', 'drawing', 'quickdraw', 'tu-berlin', 'tu_berlin', 'doodle']
PHOTO_KEYWORDS  = ['photo', 'image', 'real', 'face', 'natural']

domain_a_path = None   # Sketches
domain_b_path = None   # Photos

# ── Try auto-detection ───────────────────────────────────────────────────────
for name, path in datasets.items():
    low = name.lower()
    # Check sub-folders first (handles sketchy-dataset/ with sketch/ & photo/)
    subdirs = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
    for sd in subdirs:
        sdl = sd.lower()
        if any(kw in sdl for kw in SKETCH_KEYWORDS) and domain_a_path is None:
            domain_a_path = os.path.join(path, sd)
        if any(kw in sdl for kw in PHOTO_KEYWORDS) and domain_b_path is None:
            domain_b_path = os.path.join(path, sd)
    # Top-level match
    if any(kw in low for kw in SKETCH_KEYWORDS) and domain_a_path is None:
        domain_a_path = path
    if any(kw in low for kw in PHOTO_KEYWORDS) and domain_b_path is None:
        domain_b_path = path

# ── Fallback: first two datasets found ───────────────────────────────────────
if domain_a_path is None and len(datasets) >= 1:
    domain_a_path = list(datasets.values())[0]
if domain_b_path is None and len(datasets) >= 2:
    domain_b_path = list(datasets.values())[1]

print(f'Domain A (Sketches) : {domain_a_path}')
print(f'Domain B (Photos)   : {domain_b_path}')

assert domain_a_path is not None and os.path.exists(domain_a_path), \
    f'Domain A path not found: {domain_a_path}  →  Add a sketch dataset to the Kaggle notebook.'
assert domain_b_path is not None and os.path.exists(domain_b_path), \
    f'Domain B path not found: {domain_b_path}  →  Add a photo dataset to the Kaggle notebook.'

In [ ]:
# ── 3.4  Dataset class ──────────────────────────────────────────────────────

class UnpairedImageDataset(Dataset):
    """Loads unpaired images from Domain A and Domain B.
    
    • Domain A images are converted to grayscale (3-channel) to mimic sketches.
    • Domain B images stay as RGB photos.
    • Both are resized to IMAGE_SIZE × IMAGE_SIZE and normalised to [-1, 1].
    """

    def __init__(self, path_a, path_b, image_size=128, max_images=None, augment=True):
        self.images_a = collect_images(path_a, max_images)
        self.images_b = collect_images(path_b, max_images)
        assert len(self.images_a) > 0, f'No images found in Domain A: {path_a}'
        assert len(self.images_b) > 0, f'No images found in Domain B: {path_b}'

        # ── Transforms ─────────────────────────────────────────────────────
        common = [
            transforms.Resize((image_size, image_size), transforms.InterpolationMode.BILINEAR),
        ]
        if augment:
            common.append(transforms.RandomHorizontalFlip(p=0.5))

        self.tf_sketch = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            *common,
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
        self.tf_photo = transforms.Compose([
            *common,
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

        self.size = image_size
        print(f'Domain A (sketch) : {len(self.images_a):,} images')
        print(f'Domain B (photo)  : {len(self.images_b):,} images')

    def __len__(self):
        return max(len(self.images_a), len(self.images_b))

    def __getitem__(self, idx):
        idx_a = idx % len(self.images_a)
        idx_b = random.randint(0, len(self.images_b) - 1)  # unpaired

        # Domain A — sketch
        try:
            img_a = Image.open(self.images_a[idx_a]).convert('RGB')
            img_a = self.tf_sketch(img_a)
        except Exception:
            img_a = torch.zeros(3, self.size, self.size)  # safe fallback

        # Domain B — photo
        try:
            img_b = Image.open(self.images_b[idx_b]).convert('RGB')
            img_b = self.tf_photo(img_b)
        except Exception:
            img_b = torch.zeros(3, self.size, self.size)

        return {'A': img_a, 'B': img_b}

In [ ]:
# ── 3.5  Create DataLoader ─────────────────────────────────────────────────

train_dataset = UnpairedImageDataset(
    domain_a_path, domain_b_path,
    image_size=IMAGE_SIZE,
    max_images=MAX_DATASET,
    augment=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

print(f'\nDataLoader ready  –  {len(train_dataset):,} samples, '
      f'{len(train_loader):,} batches per epoch')

In [ ]:
# ── 3.6  Preview a batch ────────────────────────────────────────────────────

sample = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(min(4, BATCH_SIZE)):
    # Domain A
    img_a = sample['A'][i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[0, i].imshow(np.clip(img_a, 0, 1))
    axes[0, i].set_title(f'Sketch {i}')
    axes[0, i].axis('off')
    # Domain B
    img_b = sample['B'][i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[1, i].imshow(np.clip(img_b, 0, 1))
    axes[1, i].set_title(f'Photo {i}')
    axes[1, i].axis('off')
fig.suptitle('Sample Batch — Domain A (Sketches) vs Domain B (Photos)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_batch.png'), dpi=100)
plt.show()

---
## 4 · Model Architecture

In [ ]:
# ── 4.1  Residual Block ─────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    """Two-conv residual block with InstanceNorm + ReflectionPad."""

    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)

In [ ]:
# ── 4.2  ResNet Generator ───────────────────────────────────────────────────

class ResNetGenerator(nn.Module):
    """ResNet-based generator for CycleGAN.

    Structure:
        c7s1-64  →  d128  →  d256  →  R256×n_blocks  →  u128  →  u64  →  c7s1-3
    """

    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_blocks=6):
        super().__init__()

        # ── Initial: c7s1-ngf ────────────────────────────────────────────
        self.initial = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(True),
        )

        # ── Downsampling: d(ngf*2), d(ngf*4) ────────────────────────────
        self.down = nn.Sequential(
            nn.Conv2d(ngf,   ngf*2, 3, stride=2, padding=1),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(True),
            nn.Conv2d(ngf*2, ngf*4, 3, stride=2, padding=1),
            nn.InstanceNorm2d(ngf*4),
            nn.ReLU(True),
        )

        # ── Residual blocks ──────────────────────────────────────────────
        self.res = nn.Sequential(*[ResidualBlock(ngf*4) for _ in range(n_blocks)])

        # ── Upsampling: u(ngf*2), u(ngf) ────────────────────────────────
        self.up = nn.Sequential(
            nn.ConvTranspose2d(ngf*4, ngf*2, 3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(True),
        )

        # ── Final: c7s1-out_ch ───────────────────────────────────────────
        self.final = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, out_ch, 7),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.initial(x)
        x = self.down(x)
        x = self.res(x)
        x = self.up(x)
        return self.final(x)

In [ ]:
# ── 4.3  PatchGAN Discriminator ─────────────────────────────────────────────

class PatchGANDiscriminator(nn.Module):
    """70×70 PatchGAN discriminator.

    Classifies 70×70 overlapping image patches as real/fake.
    """

    def __init__(self, in_ch=3, ndf=64):
        super().__init__()

        def block(in_c, out_c, stride=2, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, stride=stride, padding=1)]
            if norm:
                layers.append(nn.InstanceNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers

        self.model = nn.Sequential(
            *block(in_ch, ndf,   stride=2, norm=False),  # C64
            *block(ndf,   ndf*2, stride=2),              # C128
            *block(ndf*2, ndf*4, stride=2),              # C256
            *block(ndf*4, ndf*8, stride=1),              # C512 (stride=1)
            nn.Conv2d(ndf*8, 1, 4, stride=1, padding=1), # Patch output
        )

    def forward(self, x):
        return self.model(x)

---
## 5 · Loss Functions

In [ ]:
# ── 5.1  CycleGAN Loss Module ──────────────────────────────────────────────

class CycleGANLoss(nn.Module):
    """Bundles adversarial, cycle-consistency, and identity losses."""

    def __init__(self, lambda_cycle=10.0, lambda_identity=5.0):
        super().__init__()
        self.lambda_cycle    = lambda_cycle
        self.lambda_identity = lambda_identity
        self.mse  = nn.MSELoss()    # LSGAN adversarial loss
        self.l1   = nn.L1Loss()     # Cycle & Identity

    # ── A. Adversarial (LSGAN) ───────────────────────────────────────────
    def adversarial(self, pred, is_real):
        target = torch.ones_like(pred) if is_real else torch.zeros_like(pred)
        return self.mse(pred, target)

    # ── B. Cycle Consistency ─────────────────────────────────────────────
    def cycle(self, real, reconstructed):
        return self.l1(real, reconstructed) * self.lambda_cycle

    # ── C. Identity ──────────────────────────────────────────────────────
    def identity(self, real, same):
        return self.l1(real, same) * self.lambda_identity

print('CycleGANLoss ready')

---
## 6 · Metrics (SSIM & PSNR)

In [ ]:
# ── 6  Calculate SSIM and PSNR between two batches ─────────────────────────

def calculate_metrics(real: torch.Tensor, generated: torch.Tensor):
    """Compute mean SSIM and PSNR over a batch.
    
    Both tensors must be in [-1, 1] range (C, H, W).
    Returns (mean_ssim, mean_psnr).
    """
    real_np = (real.detach().cpu().numpy() + 1.0) / 2.0       # → [0, 1]
    gen_np  = (generated.detach().cpu().numpy() + 1.0) / 2.0

    ssim_vals, psnr_vals = [], []
    for i in range(real_np.shape[0]):
        r = np.clip(real_np[i].transpose(1, 2, 0), 0, 1)   # (H, W, C)
        g = np.clip(gen_np[i].transpose(1, 2, 0),  0, 1)
        ssim_vals.append(ssim(r, g, data_range=1.0, channel_axis=2))
        psnr_vals.append(psnr(r, g, data_range=1.0))
    return float(np.mean(ssim_vals)), float(np.mean(psnr_vals))

print('calculate_metrics ready')

---
## 7 · Model Initialisation

In [ ]:
# ── 7.1  Weight initialisation ─────────────────────────────────────────────

def init_weights(m):
    """Initialise Conv and InstanceNorm layers."""
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif 'InstanceNorm' in classname and m.weight is not None:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

# ── 7.2  Build models ──────────────────────────────────────────────────────
G_AB = ResNetGenerator(3, 3, NGF, RESNET_BLOCKS).to(device)   # Sketch → Photo
G_BA = ResNetGenerator(3, 3, NGF, RESNET_BLOCKS).to(device)   # Photo  → Sketch
D_A  = PatchGANDiscriminator(3, NDF).to(device)               # Classifies Sketch
D_B  = PatchGANDiscriminator(3, NDF).to(device)               # Classifies Photo

G_AB.apply(init_weights)
G_BA.apply(init_weights)
D_A.apply(init_weights)
D_B.apply(init_weights)

# ── Multi-GPU ────────────────────────────────────────────────────────────────
if NUM_GPUS > 1:
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A  = nn.DataParallel(D_A)
    D_B  = nn.DataParallel(D_B)
    print(f'DataParallel enabled ({NUM_GPUS} GPUs)')

# ── 7.3  Optimisers ─────────────────────────────────────────────────────────
opt_G   = optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                     lr=LR, betas=BETAS)
opt_D_A = optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
opt_D_B = optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)

# ── 7.4  LR Schedulers (linear decay from DECAY_START) ──────────────────────
def lr_lambda(epoch):
    """Linear decay starting from DECAY_START."""
    if epoch < DECAY_START:
        return 1.0
    return 1.0 - (epoch - DECAY_START) / (NUM_EPOCHS - DECAY_START + 1)

sched_G   = optim.lr_scheduler.LambdaLR(opt_G,   lr_lambda)
sched_D_A = optim.lr_scheduler.LambdaLR(opt_D_A, lr_lambda)
sched_D_B = optim.lr_scheduler.LambdaLR(opt_D_B, lr_lambda)

# ── 7.5  Loss & Scalers ──────────────────────────────────────────────────────
criterion = CycleGANLoss(LAMBDA_CYCLE, LAMBDA_IDENTITY)
scaler_G  = torch.amp.GradScaler('cuda')   # separate scaler per optimizer family
scaler_D  = torch.amp.GradScaler('cuda')

# ── Count parameters ─────────────────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nG_AB params : {count_params(G_AB):>10,}')
print(f'G_BA params : {count_params(G_BA):>10,}')
print(f'D_A  params : {count_params(D_A):>10,}')
print(f'D_B  params : {count_params(D_B):>10,}')
print('\nModels, optimizers, schedulers initialised ✓')

---
## 8 · Image Replay Buffer

In [ ]:
# ── 8  Replay buffer (stabilises discriminator training) ───────────────────

class ImageBuffer:
    """Stores previously generated images and probabilistically returns
    a mix of new and buffered images to the discriminator.
    """

    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, images: torch.Tensor) -> torch.Tensor:
        """images: (B, C, H, W) tensor.  Returns tensor of same shape."""
        if self.max_size == 0:
            return images

        result = []
        for img in images:                       # img shape: (C, H, W)
            img = img.detach()
            if len(self.data) < self.max_size:
                self.data.append(img.clone())
                result.append(img)
            elif random.random() > 0.5:
                idx = random.randint(0, self.max_size - 1)
                old = self.data[idx].clone()
                self.data[idx] = img.clone()
                result.append(old)
            else:
                result.append(img)

        return torch.stack(result, dim=0)         # (B, C, H, W)

fake_A_buffer = ImageBuffer(BUFFER_SIZE)
fake_B_buffer = ImageBuffer(BUFFER_SIZE)
print('Replay buffers ready')

---
## 9 · Training Loop

In [ ]:
# ── 9  Full training function ──────────────────────────────────────────────

def train_cyclegan(loader, epochs):
    """
    Train CycleGAN with mixed-precision on dual GPUs.

    Forward pass per iteration:
        1. G_AB(sketch)  → fake_photo       ;  G_BA(fake_photo)  → rec_sketch
        2. G_BA(photo)   → fake_sketch      ;  G_AB(fake_sketch) → rec_photo
        3. Identity:  G_BA(sketch), G_AB(photo)
    """
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()

    history = {
        'G': [], 'D_A': [], 'D_B': [],
        'Cycle': [], 'Identity': [],
        'SSIM': [], 'PSNR': [],
    }

    for epoch in range(epochs):
        accum = defaultdict(float)
        pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}', leave=True)

        for batch in pbar:
            real_A = batch['A'].to(device)   # sketches
            real_B = batch['B'].to(device)   # photos

            # ═══════════════════════════════════════════════════════════════
            # ── GENERATOR STEP ────────────────────────────────────────────
            # ═══════════════════════════════════════════════════════════════
            opt_G.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                # Forward translations
                fake_B  = G_AB(real_A)              # Sketch → Photo
                fake_A  = G_BA(real_B)              # Photo  → Sketch
                rec_A   = G_BA(fake_B)              # Photo  → Sketch (cycle)
                rec_B   = G_AB(fake_A)              # Sketch → Photo  (cycle)
                idt_A   = G_BA(real_A)              # Sketch → Sketch (identity)
                idt_B   = G_AB(real_B)              # Photo  → Photo  (identity)

                # Adversarial losses — generators want discriminators to say "real"
                loss_gan_AB  = criterion.adversarial(D_B(fake_B), True)
                loss_gan_BA  = criterion.adversarial(D_A(fake_A), True)

                # Cycle consistency
                loss_cyc_A   = criterion.cycle(real_A, rec_A)
                loss_cyc_B   = criterion.cycle(real_B, rec_B)

                # Identity
                loss_idt_A   = criterion.identity(real_A, idt_A)
                loss_idt_B   = criterion.identity(real_B, idt_B)

                loss_G = (loss_gan_AB + loss_gan_BA
                          + loss_cyc_A + loss_cyc_B
                          + loss_idt_A + loss_idt_B)

            scaler_G.scale(loss_G).backward()
            scaler_G.step(opt_G)
            scaler_G.update()

            # ═══════════════════════════════════════════════════════════════
            # ── DISCRIMINATOR A STEP ──────────────────────────────────────
            # ═══════════════════════════════════════════════════════════════
            opt_D_A.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                buf_fake_A = fake_A_buffer.push_and_pop(fake_A.detach())
                loss_dA = 0.5 * (
                    criterion.adversarial(D_A(real_A),    True) +
                    criterion.adversarial(D_A(buf_fake_A), False)
                )

            scaler_D.scale(loss_dA).backward()
            scaler_D.step(opt_D_A)
            scaler_D.update()

            # ═══════════════════════════════════════════════════════════════
            # ── DISCRIMINATOR B STEP ──────────────────────────────────────
            # ═══════════════════════════════════════════════════════════════
            opt_D_B.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                buf_fake_B = fake_B_buffer.push_and_pop(fake_B.detach())
                loss_dB = 0.5 * (
                    criterion.adversarial(D_B(real_B),    True) +
                    criterion.adversarial(D_B(buf_fake_B), False)
                )

            scaler_D.scale(loss_dB).backward()
            scaler_D.step(opt_D_B)
            scaler_D.update()

            # ── Accumulate ────────────────────────────────────────────────
            accum['G']        += loss_G.item()
            accum['D_A']      += loss_dA.item()
            accum['D_B']      += loss_dB.item()
            accum['Cycle']    += (loss_cyc_A + loss_cyc_B).item()
            accum['Identity'] += (loss_idt_A + loss_idt_B).item()

            # Per-batch metrics (on reconstructed vs real — not every batch to save time)
            with torch.no_grad():
                s, p = calculate_metrics(real_A, rec_A)
                accum['SSIM'] += s
                accum['PSNR'] += p

            pbar.set_postfix(G=f"{loss_G.item():.3f}",
                             DA=f"{loss_dA.item():.3f}",
                             DB=f"{loss_dB.item():.3f}")

        # ── Epoch averages ────────────────────────────────────────────────
        n = len(loader)
        for k in accum:
            accum[k] /= n
            history[k].append(accum[k])

        # ── Step schedulers ───────────────────────────────────────────────
        sched_G.step()
        sched_D_A.step()
        sched_D_B.step()

        cur_lr = opt_G.param_groups[0]['lr']
        print(f"  ↳ G={accum['G']:.4f}  D_A={accum['D_A']:.4f}  D_B={accum['D_B']:.4f}  "
              f"Cyc={accum['Cycle']:.4f}  Idt={accum['Identity']:.4f}  "
              f"SSIM={accum['SSIM']:.4f}  PSNR={accum['PSNR']:.2f}  LR={cur_lr:.6f}")

        # ── Checkpoint ────────────────────────────────────────────────────
        if (epoch + 1) % CKPT_INTERVAL == 0 or (epoch + 1) == epochs:
            ckpt = {
                'epoch': epoch + 1,
                'G_AB': G_AB.state_dict(),
                'G_BA': G_BA.state_dict(),
                'D_A':  D_A.state_dict(),
                'D_B':  D_B.state_dict(),
                'opt_G':   opt_G.state_dict(),
                'opt_D_A': opt_D_A.state_dict(),
                'opt_D_B': opt_D_B.state_dict(),
            }
            torch.save(ckpt, os.path.join(CHECKPOINT_DIR, f'cyclegan_epoch{epoch+1}.pth'))
            print(f'  💾  Checkpoint saved: epoch {epoch+1}')

        # ── Quick visual every 10 epochs ──────────────────────────────────
        if (epoch + 1) % 10 == 0:
            quick_visual(epoch + 1)

        # ── Free memory ───────────────────────────────────────────────────
        torch.cuda.empty_cache()

    return history

print('train_cyclegan ready')

---
## 10 · Visualisation Utilities

In [ ]:
# ── 10.1  Quick visual during training ─────────────────────────────────────

def quick_visual(epoch_num, n=4):
    """Generate a grid: Input Sketch | Fake Photo | Rec Sketch (and vice versa)."""
    G_AB.eval(); G_BA.eval()
    batch = next(iter(train_loader))
    real_A = batch['A'][:n].to(device)
    real_B = batch['B'][:n].to(device)

    with torch.no_grad(), torch.amp.autocast('cuda'):
        fake_B = G_AB(real_A)
        rec_A  = G_BA(fake_B)
        fake_A = G_BA(real_B)
        rec_B  = G_AB(fake_A)

    # Concat: rows = [real_A, fake_B, rec_A, real_B, fake_A, rec_B]
    imgs = torch.cat([real_A, fake_B, rec_A, real_B, fake_A, rec_B], dim=0)
    grid = make_grid(imgs, nrow=n, normalize=True, value_range=(-1, 1))

    plt.figure(figsize=(n * 3, 18))
    plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
    plt.title(f'Epoch {epoch_num}  —  Rows: Sketch→Photo→RecSketch | Photo→Sketch→RecPhoto', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'visual_epoch{epoch_num}.png'), dpi=100)
    plt.show()

    G_AB.train(); G_BA.train()

print('quick_visual ready')

In [ ]:
# ── 10.2  Plot all training losses ─────────────────────────────────────────

def plot_losses(history):
    """Three-panel loss plots as required by the assignment."""
    epochs = range(1, len(history['G']) + 1)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1 — Generator Loss vs Epochs
    axes[0, 0].plot(epochs, history['G'], 'b-', lw=2)
    axes[0, 0].set_title('Generator Loss', fontsize=13)
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True, alpha=0.3)

    # 2 — Discriminator Loss vs Epochs
    axes[0, 1].plot(epochs, history['D_A'], 'r-', lw=2, label='D_A (Sketch)')
    axes[0, 1].plot(epochs, history['D_B'], 'g-', lw=2, label='D_B (Photo)')
    axes[0, 1].set_title('Discriminator Loss', fontsize=13)
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    # 3 — Cycle Consistency Loss vs Epochs
    axes[0, 2].plot(epochs, history['Cycle'], 'm-', lw=2)
    axes[0, 2].set_title('Cycle Consistency Loss', fontsize=13)
    axes[0, 2].set_xlabel('Epoch'); axes[0, 2].set_ylabel('Loss')
    axes[0, 2].grid(True, alpha=0.3)

    # 4 — Identity Loss
    axes[1, 0].plot(epochs, history['Identity'], 'c-', lw=2)
    axes[1, 0].set_title('Identity Loss', fontsize=13)
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Loss')
    axes[1, 0].grid(True, alpha=0.3)

    # 5 — SSIM
    axes[1, 1].plot(epochs, history['SSIM'], 'orange', lw=2)
    axes[1, 1].set_title('SSIM (higher ↑ is better)', fontsize=13)
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('SSIM')
    axes[1, 1].grid(True, alpha=0.3)

    # 6 — PSNR
    axes[1, 2].plot(epochs, history['PSNR'], 'brown', lw=2)
    axes[1, 2].set_title('PSNR (higher ↑ is better)', fontsize=13)
    axes[1, 2].set_xlabel('Epoch'); axes[1, 2].set_ylabel('PSNR (dB)')
    axes[1, 2].grid(True, alpha=0.3)

    fig.suptitle('CycleGAN Training Metrics', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'cyclegan_losses.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('plot_losses ready')

In [ ]:
# ── 10.3  Qualitative Evaluation (≥ 5 examples) ───────────────────────────

def visualize_results(loader, n_samples=5):
    """Show n_samples  ×  3 rows:
       Row 1: Input Sketch
       Row 2: Generated Photo
       Row 3: Reconstructed Sketch
    """
    G_AB.eval(); G_BA.eval()
    batch = next(iter(loader))
    real_A = batch['A'][:n_samples].to(device)
    real_B = batch['B'][:n_samples].to(device)

    with torch.no_grad(), torch.amp.autocast('cuda'):
        fake_B = G_AB(real_A)
        rec_A  = G_BA(fake_B)
        fake_A = G_BA(real_B)
        rec_B  = G_AB(fake_A)

    def to_np(t):
        return (t.cpu().permute(0, 2, 3, 1).numpy() + 1) / 2

    rA, fB, rcA = to_np(real_A), to_np(fake_B), to_np(rec_A)
    rB, fA, rcB = to_np(real_B), to_np(fake_A), to_np(rec_B)

    fig, axes = plt.subplots(2, n_samples * 3, figsize=(n_samples * 9, 6))
    titles_top = ['Sketch', 'Gen Photo', 'Rec Sketch'] * n_samples
    titles_bot = ['Photo',  'Gen Sketch', 'Rec Photo'] * n_samples

    for i in range(n_samples):
        axes[0, i*3 + 0].imshow(np.clip(rA[i], 0, 1));  axes[0, i*3 + 0].set_title(titles_top[i*3 + 0])
        axes[0, i*3 + 1].imshow(np.clip(fB[i], 0, 1));  axes[0, i*3 + 1].set_title(titles_top[i*3 + 1])
        axes[0, i*3 + 2].imshow(np.clip(rcA[i], 0, 1)); axes[0, i*3 + 2].set_title(titles_top[i*3 + 2])

        axes[1, i*3 + 0].imshow(np.clip(rB[i], 0, 1));  axes[1, i*3 + 0].set_title(titles_bot[i*3 + 0])
        axes[1, i*3 + 1].imshow(np.clip(fA[i], 0, 1));  axes[1, i*3 + 1].set_title(titles_bot[i*3 + 1])
        axes[1, i*3 + 2].imshow(np.clip(rcB[i], 0, 1)); axes[1, i*3 + 2].set_title(titles_bot[i*3 + 2])

    for ax in axes.flat:
        ax.axis('off')

    fig.suptitle('Qualitative Results — Input ↔ Translated ↔ Reconstructed', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'qualitative_results.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('visualize_results ready')

---
## 11 · Train!

In [ ]:
print('=' * 70)
print('  CycleGAN Training — Starting')
print('=' * 70)

losses_history = train_cyclegan(train_loader, NUM_EPOCHS)

print('\n' + '=' * 70)
print('  Training Complete ✓')
print('=' * 70)

---
## 12 · Save Final Models

In [ ]:
# ── 12.1  Save combined checkpoint ─────────────────────────────────────────
final_ckpt = {
    'G_AB': G_AB.state_dict(),
    'G_BA': G_BA.state_dict(),
    'D_A':  D_A.state_dict(),
    'D_B':  D_B.state_dict(),
}
torch.save(final_ckpt, os.path.join(CHECKPOINT_DIR, 'cyclegan_final.pth'))
print('✓ Combined checkpoint saved → cyclegan_final.pth')

# ── 12.2  Save individual generators (for Gradio app) ─────────────────────
# Strip DataParallel wrapper if present
def unwrap_state_dict(model):
    if isinstance(model, nn.DataParallel):
        return model.module.state_dict()
    return model.state_dict()

torch.save(unwrap_state_dict(G_AB), os.path.join(CHECKPOINT_DIR, 'cyclegan_G_AB_final.pth'))
torch.save(unwrap_state_dict(G_BA), os.path.join(CHECKPOINT_DIR, 'cyclegan_G_BA_final.pth'))
print('✓ G_AB saved → cyclegan_G_AB_final.pth  (Sketch → Photo)')
print('✓ G_BA saved → cyclegan_G_BA_final.pth  (Photo  → Sketch)')
print(f'\nAll files in {CHECKPOINT_DIR}:')
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    sz = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1e6
    print(f'  {f:40s}  {sz:6.1f} MB')

---
## 13 · Training Logs & Plots

In [ ]:
# ── 13.1  Loss plots ──────────────────────────────────────────────────────
plot_losses(losses_history)

In [ ]:
# ── 13.2  Print final metrics ─────────────────────────────────────────────
print('\n' + '─' * 50)
print('Final Epoch Metrics')
print('─' * 50)
for key in losses_history:
    vals = losses_history[key]
    print(f'  {key:10s}  last={vals[-1]:.4f}   best={min(vals) if key not in ("SSIM","PSNR") else max(vals):.4f}')
print('─' * 50)

---
## 14 · Quantitative Evaluation (SSIM & PSNR)

In [ ]:
# ── 14  Per-sample SSIM/PSNR evaluation over many batches ─────────────────

G_AB.eval(); G_BA.eval()

all_ssim_ab, all_psnr_ab = [], []
all_ssim_ba, all_psnr_ba = [], []

eval_batches = min(50, len(train_loader))
print(f'Evaluating over {eval_batches} batches ...')

for i, batch in enumerate(train_loader):
    if i >= eval_batches:
        break
    real_A = batch['A'].to(device)
    real_B = batch['B'].to(device)

    with torch.no_grad(), torch.amp.autocast('cuda'):
        fake_B = G_AB(real_A)
        rec_A  = G_BA(fake_B)
        fake_A = G_BA(real_B)
        rec_B  = G_AB(fake_A)

    s, p = calculate_metrics(real_A, rec_A)
    all_ssim_ab.append(s); all_psnr_ab.append(p)

    s, p = calculate_metrics(real_B, rec_B)
    all_ssim_ba.append(s); all_psnr_ba.append(p)

print('\n' + '═' * 60)
print('  Quantitative Evaluation — Cycle Reconstruction')
print('═' * 60)
print(f'  A→B→A  SSIM : {np.mean(all_ssim_ab):.4f}  ±  {np.std(all_ssim_ab):.4f}')
print(f'  A→B→A  PSNR : {np.mean(all_psnr_ab):.2f}   ±  {np.std(all_psnr_ab):.2f} dB')
print(f'  B→A→B  SSIM : {np.mean(all_ssim_ba):.4f}  ±  {np.std(all_ssim_ba):.4f}')
print(f'  B→A→B  PSNR : {np.mean(all_psnr_ba):.2f}   ±  {np.std(all_psnr_ba):.2f} dB')
print('═' * 60)

---
## 15 · Qualitative Results (≥ 5 examples)

In [ ]:
visualize_results(train_loader, n_samples=5)

In [ ]:
# ── Additional visual: Input vs Translated vs Reconstructed ───────────────

G_AB.eval(); G_BA.eval()
batch = next(iter(train_loader))

real_A = batch['A'][:8].to(device)
real_B = batch['B'][:8].to(device)

with torch.no_grad(), torch.amp.autocast('cuda'):
    fake_B = G_AB(real_A)
    rec_A  = G_BA(fake_B)

# Save grids
save_image(real_A * 0.5 + 0.5, os.path.join(OUTPUT_DIR, 'final_input_sketches.png'), nrow=8)
save_image(fake_B * 0.5 + 0.5, os.path.join(OUTPUT_DIR, 'final_generated_photos.png'), nrow=8)
save_image(rec_A  * 0.5 + 0.5, os.path.join(OUTPUT_DIR, 'final_reconstructed_sketches.png'), nrow=8)

# Display
fig, axes = plt.subplots(3, 1, figsize=(20, 8))
for ax, imgs, title in zip(axes,
    [real_A, fake_B, rec_A],
    ['Input Sketches', 'Generated Photos', 'Reconstructed Sketches']):
    grid = make_grid(imgs, nrow=8, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.cpu().permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=13)
    ax.axis('off')
plt.suptitle('Input ↔ Translated ↔ Reconstructed', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'final_comparison.png'), dpi=150)
plt.show()

---
## 16 · Summary

In [ ]:
print('=' * 70)
print('  CycleGAN Training Summary')
print('=' * 70)
print(f'  Device          : {device}  ({NUM_GPUS} GPU(s))')
print(f'  Image Size      : {IMAGE_SIZE}×{IMAGE_SIZE}')
print(f'  Batch Size      : {BATCH_SIZE}')
print(f'  Epochs          : {NUM_EPOCHS}')
print(f'  ResNet Blocks   : {RESNET_BLOCKS}')
print(f'  λ Cycle         : {LAMBDA_CYCLE}')
print(f'  λ Identity      : {LAMBDA_IDENTITY}')
print(f'  LR              : {LR} → linear decay from epoch {DECAY_START}')
print(f'  Mixed Precision : ✓')
print(f'  G_AB params     : {count_params(G_AB):,}')
print(f'  G_BA params     : {count_params(G_BA):,}')
print(f'  D_A  params     : {count_params(D_A):,}')
print(f'  D_B  params     : {count_params(D_B):,}')
print(f'  Output Dir      : {OUTPUT_DIR}')
print(f'  Checkpoint Dir  : {CHECKPOINT_DIR}')
print('=' * 70)
print('\n✅  Notebook complete.  Download .pth files from checkpoint dir')
print('    and place in Task3/ folder for the Gradio app.')